In [ ]:
import pandas as pd
import numpy as np
import ast

import torch
from torch import nn

from fastai.collab import *
from fastai.tabular.all import *


data = pd.read_csv('movies.csv')

print("Dataset shape:", data.shape)

print("\nKeywords column type:")
print(type(data["Keywords"].iloc[0]))

print("\nFirst movie keywords:")
print(data["Keywords"].iloc[0])

print("\nTrain/test splits:")
print(data["split"].value_counts(dropna=False))


In [ ]:
def extract_keywords(value):

    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(value, list):
        return []

    return [
        item["name"]
        for item in value
        if isinstance(item, dict) and "name" in item
    ]


data["keyword_list"] = data["Keywords"].apply(
    extract_keywords
)

data[["title", "keyword_list", "revenue"]].head()

In [ ]:
from collections import Counter

keyword_counts = Counter(
    keyword
    for keywords in data["keyword_list"]
    for keyword in keywords
)

print("Number of unique keywords:", len(keyword_counts))

print("\nMost common keywords:")
print(keyword_counts.most_common(20))

print("\nMovies without keywords:")
print((data["keyword_list"].str.len() == 0).sum())


In [ ]:
from sklearn.model_selection import train_test_split

train_data = data[data["split"] == "train"].copy()

valid_data = data[
    data["split"].isin(["valid", "validation"])
].copy()

# If no validation partition exists, create one
# using only training movies.
if valid_data.empty:
    train_data, valid_data = train_test_split(
        train_data,
        test_size=0.2,
        random_state=42
    )

print("Training movies:", len(train_data))
print("Validation movies:", len(valid_data))

for df in [train_data, valid_data]:
    df["revenue"] = pd.to_numeric(
        df["revenue"], errors="coerce"
    )

    df["target"] = np.log1p(
        df["revenue"].clip(lower=0)
    )

    df.dropna(subset=["target"], inplace=True)

from collections import Counter

keyword_counts = Counter(
    keyword
    for keywords in train_data["keyword_list"]
    for keyword in keywords
)

min_frequency = 2

vocab = {
    keyword: idx + 1
    for idx, (keyword, count) in enumerate(
        (k, c)
        for k, c in keyword_counts.items()
        if c >= min_frequency
    )
}

vocab_size = len(vocab) + 1

print("Vocabulary size:", vocab_size)

def encode_keywords(keywords):
    return [
        vocab[keyword]
        for keyword in keywords
        if keyword in vocab
    ]


train_data["keyword_ids"] = (
    train_data["keyword_list"].apply(encode_keywords)
)

valid_data["keyword_ids"] = (
    valid_data["keyword_list"].apply(encode_keywords)
)

print(train_data[["keyword_list", "keyword_ids"]].head())




In [ ]:
embedding_dim = 16

keyword_embedding = nn.EmbeddingBag(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    mode="mean",
    padding_idx=0
)

movie_keywords = train_data["keyword_ids"].iloc[0]

if not movie_keywords:
    movie_keywords = [0]

ids = torch.tensor(movie_keywords, dtype=torch.long)

offsets = torch.tensor([0], dtype=torch.long)

movie_embedding = keyword_embedding(ids, offsets)

print("Keyword indices:", movie_keywords)
print("Embedding shape:", movie_embedding.shape)
print("Embedding:", movie_embedding)

In [ ]:
from torch import nn

class KeywordRevenueModel(nn.Module):

    def __init__(self, embedding):
        super().__init__()

        self.embedding = embedding

        self.regressor = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, ids, offsets):

        # Average keyword embeddings
        z = self.embedding(ids, offsets)

        # Predict log revenue
        prediction = self.regressor(z)

        return prediction.squeeze(-1)


model = KeywordRevenueModel(keyword_embedding)

from torch.utils.data import Dataset, DataLoader

class MovieDataset(Dataset):

    def __init__(self, df):
        self.keywords = df["keyword_ids"].tolist()
        self.targets = df["target"].to_numpy(
            dtype="float32"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.keywords[idx], self.targets[idx]

import torch

def collate_movies(batch):

    keyword_lists, targets = zip(*batch)

    # Replace empty lists with padding token
    keyword_lists = [
        ids if len(ids) > 0 else [0]
        for ids in keyword_lists
    ]

    lengths = [len(ids) for ids in keyword_lists]

    offsets = torch.tensor(
        [0] + list(np.cumsum(lengths[:-1])),
        dtype=torch.long
    )

    ids = torch.tensor(
        [k for movie in keyword_lists for k in movie],
        dtype=torch.long
    )

    targets = torch.tensor(
        targets,
        dtype=torch.float32
    )

    return ids, offsets, targets

train_dl = DataLoader(
    MovieDataset(train_data),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_movies
)

valid_dl = DataLoader(
    MovieDataset(valid_data),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_movies
)

ids, offsets, targets = next(iter(train_dl))

print("Keyword tensor:", ids.shape)
print("Offsets:", offsets.shape)
print("Targets:", targets.shape)

In [ ]:
from fastai.data.core import DataLoaders
from fastai.learner import Learner

dls = DataLoaders(train_dl, valid_dl)

# Two model inputs: ids and offsets
dls.n_inp = 2

learn = Learner(
    dls,
    model,
    loss_func=nn.MSELoss()
)

learn.lr_find()

learn.fit_one_cycle(
    10,
    1e-3,
    wd=0.1
)
learn.recorder.plot_loss()

model.eval()

with torch.no_grad():

    ids, offsets, targets = next(iter(valid_dl))

    device = next(model.parameters()).device

    predictions = model(
        ids.to(device),
        offsets.to(device)
    ).cpu()

comparison = pd.DataFrame({
    "Actual log revenue": targets.numpy(),
    "Predicted log revenue": predictions.numpy()
})

comparison.head(10)

from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(
        comparison["Actual log revenue"],
        comparison["Predicted log revenue"]
    )
)

print("Batch validation RMSE:", rmse)

learn.model.eval()

keyword_vectors = (
    learn.model.embedding.weight
    .detach()
    .cpu()
    .numpy()
)

print("Embedding matrix shape:", keyword_vectors.shape)




In [ ]:
# Use our existing trained vocabulary
data["keyword_ids"] = data["keyword_list"].apply(
    encode_keywords
)

print(data[["title", "keyword_ids"]].head())

def embed_movie(keyword_ids, model):

    model.eval()

    # Handle movies without recognized keywords
    if len(keyword_ids) == 0:
        keyword_ids = [0]

    device = next(model.parameters()).device

    ids = torch.tensor(
        keyword_ids,
        dtype=torch.long,
        device=device
    )

    offsets = torch.tensor(
        [0],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        embedding = model.embedding(
            ids, offsets
        )

    return embedding.cpu().numpy().flatten()

example_ids = data["keyword_ids"].iloc[0]

example_embedding = embed_movie(
    example_ids,
    learn.model
)

print("Embedding shape:", example_embedding.shape)
print("Embedding vector:")
print(example_embedding)



In [ ]:
movie_embeddings = np.vstack([
    embed_movie(ids, learn.model)
    for ids in data["keyword_ids"]
])

print("Embedding matrix shape:", movie_embeddings.shape)


embedding_dim = movie_embeddings.shape[1]

embedding_columns = [
    f"keyword_emb_{i}"
    for i in range(embedding_dim)
]

keyword_embeddings_df = pd.DataFrame(
    movie_embeddings,
    columns=embedding_columns,
    index=data.index
)

keyword_embeddings_df.head()




data_embedded = pd.concat(
    [
        data.drop(
            columns=[
                "Keywords",
                "keyword_list",
                "keyword_ids"
            ],
            errors="ignore"
        ),
        keyword_embeddings_df
    ],
    axis=1
)

print("Original shape:", data.shape)
print("Embedded shape:", data_embedded.shape)


print("Number of movies:", len(data_embedded))

print("\nEmbedding columns:")
print(embedding_columns)

print("\nMissing embedding values:")
print(
    data_embedded[embedding_columns]
    .isna()
    .sum()
    .sum()
)

print("\nFirst movie:")
print(
    data_embedded[
        ["title", "revenue"] + embedding_columns
    ].head(1)
)

assert len(data_embedded) == len(data)

assert data_embedded["id"].reset_index(
    drop=True
).equals(
    data["id"].reset_index(drop=True)
)

assert not data_embedded[
    embedding_columns
].isna().any().any()



In [ ]:
output_file = "movies_with_keyword_embeddings.csv"

data_embedded.to_csv(
    output_file,
    index=False
)

print(f"Saved: {output_file}")

data_new = pd.read_csv(output_file)

print(data_new.shape)

data_new[
    ["title", "revenue"] + embedding_columns
].head()



In [ ]:
data_new = pd.read_csv(output_file)

print(data_new.shape)

data_new[ ['title', 'revenue'] + embedding_columns].head()

In [ ]:
torch.save(
    learn.model.state_dict(),
    "keyword_embedding_model.pth"
)

import json

with open("keyword_vocab.json", "w") as f:
    json.dump(vocab, f, indent=2)

with open("keyword_vocab.json") as f:
    loaded_vocab = json.load(f)

new_embedding = nn.EmbeddingBag(
    len(loaded_vocab) + 1,
    16,
    mode="mean",
    padding_idx=0
)

loaded_model = KeywordRevenueModel(new_embedding)

loaded_model.load_state_dict(
    torch.load(
        "keyword_embedding_model.pth",
        map_location="cpu",
        weights_only=True
    )
)

loaded_model.eval()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=3)

X_pca = pca.fit_transform(movie_embeddings)

fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(111, projection = '3d')

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    X_pca[:, 2],
    c=np.log1p(
        pd.to_numeric(data["revenue"], errors="coerce")
        .clip(lower=0)
        .fillna(0)
    ),
    cmap="viridis",
    alpha=0.65
)

plt.colorbar(
    scatter,
    label="Log Revenue"
)

plt.title("Movies in Learned Keyword Embedding Space")

plt.show()

# Section for ACTOR embeddings

In [ ]:
import ast

def extract_actors(value):

    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(value, list):
        return []

    return [
        str(actor).strip()
        for actor in value
        if isinstance(actor, str) and actor.strip()
    ]


data["actor_list"] = data["actor"].apply(
    extract_actors
)

data[["title", "actor_list"]].head()

train_data["actor_list"] = data.loc[
    train_data.index, "actor_list"
]

valid_data["actor_list"] = data.loc[
    valid_data.index, "actor_list"
]

from collections import Counter

actor_counts = Counter(
    actor
    for actors in train_data["actor_list"]
    for actor in set(actors)
)

print("Unique actors:", len(actor_counts))
print(actor_counts.most_common(20))


min_actor_frequency = 2

actor_vocab = {
    actor: idx + 1
    for idx, (actor, count) in enumerate(
        (a, c)
        for a, c in actor_counts.items()
        if c >= min_actor_frequency
    )
}

n_actors = len(actor_vocab) + 1

print("Actor vocabulary size:", n_actors)

In [ ]:
def encode_actors(actors):
    return [
        actor_vocab[a]
        for a in actors
        if a in actor_vocab
    ]


train_data["actor_ids"] = (
    train_data["actor_list"].apply(encode_actors)
)

valid_data["actor_ids"] = (
    valid_data["actor_list"].apply(encode_actors)
)

data["actor_ids"] = (
    data["actor_list"].apply(encode_actors)
)

actor_embedding_dim = 16

actor_embedding = nn.EmbeddingBag(
    num_embeddings=n_actors,
    embedding_dim=actor_embedding_dim,
    mode="mean",
    padding_idx=0
)

actor_model = KeywordRevenueModel(
    actor_embedding
)

actor_train = train_data.copy()
actor_valid = valid_data.copy()

actor_train["keyword_ids"] = actor_train["actor_ids"]
actor_valid["keyword_ids"] = actor_valid["actor_ids"]

actor_train_dl = DataLoader(
    MovieDataset(actor_train),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_movies
)

actor_valid_dl = DataLoader(
    MovieDataset(actor_valid),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_movies
)

actor_dls = DataLoaders(
    actor_train_dl,
    actor_valid_dl
)

actor_dls.n_inp = 2

actor_learn = Learner(
    actor_dls,
    actor_model,
    loss_func=nn.MSELoss()
)

actor_learn.fit_one_cycle(
    10,
    1e-3,
    wd=0.1
)

In [ ]:
actor_embeddings = np.vstack([
    embed_movie(ids, actor_learn.model)
    for ids in data["actor_ids"]
])

print(actor_embeddings.shape)

actor_embedding_columns = [
    f"actor_emb_{i}"
    for i in range(actor_embedding_dim)
]

actor_embeddings_df = pd.DataFrame(
    actor_embeddings,
    columns=actor_embedding_columns,
    index=data.index
)

data_embedded = pd.concat(
    [
        data_embedded.drop(
            columns=["actor"],
            errors="ignore"
        ),
        actor_embeddings_df
    ],
    axis=1
)

print(data_embedded.shape)

data_embedded.to_csv(
    "movies_keywords_actors_embedded.csv",
    index=False
)